In [10]:
!python -V

Python 3.13.9


In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error

In [12]:
import pickle

In [13]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)# 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')

    df.loc[:, 'duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.loc[:, 'duration_in_min'] = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[(df.duration_in_min >= 1) & (df.duration_in_min <= 60)]
    
    categorical = ['PULocationID','DOLocationID']
    df[categorical] = df[categorical].astype('string')
    return df
    
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [14]:
# create new cateogiral features
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [15]:
categorical = ['PU_DO'] #['PULocationID','DOLocationID']
numerical = ['trip_distance']

# 
dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [16]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [17]:
# find the model where it product least mean squared error
lr = LinearRegression()
lr.fit(X_train, y_train)

# try to predict with real value from Feb data
y_pred = lr.predict(X_val)

# test whether 
mean_squared_error(y_val, y_pred)

2.1671158166686464e+17

In [18]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [19]:
# lr = Lasso(alpha=0.001)
# lr.fit(X_train, y_train)

# y_pred = lr.predict(X_val)

# mean_squared_error(y_train, y_pred)

In [20]:
# lr = Ridge(alpha=10)
# lr.fit(X_train, y_train)

# y_pred = lr.predict(X_val)

# mean_squared_error(y_val, y_pred)